# Run the regulatory-activity workflow

This notebook starts from real H5AD inputs and writes a fresh ranking, panel, evaluation and run manifest. [Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/regulatory_section/02_SMITH_Regulatory_Activity_source.ipynb).

## Set data and output locations

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("SMITH_TUTORIAL_DATA", "data/tutorials")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("SMITH_TUTORIAL_OUTPUT", "outputs/tutorials")).expanduser().resolve()
CASE_OUTPUT = OUTPUT_ROOT / 'regulatory'
EPOCHS = int(os.environ.get("SMITH_TUTORIAL_EPOCHS", '1'))
DEVICE = os.environ.get("SMITH_TUTORIAL_DEVICE", 'cpu')
print("Input and output roots are configured. Set SMITH_TUTORIAL_DATA/OUTPUT to override them.")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## Check real input files

In [ ]:
inputs = ['regulatory_activity/elegans/splits/elegans_tf/split_1/train.h5ad', 'regulatory_activity/elegans/splits/elegans_tf/split_1/test.h5ad']
input_rows = []
for relative in inputs:
    path = DATA_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run scripts/download_tutorial_data.py first.")
    digest = sha256_file(path)
    input_rows.append({"file": relative, "bytes": path.stat().st_size, "sha256": digest})
display(pd.DataFrame(input_rows))


## Run the workflow

In [ ]:
command = [
    sys.executable, str(ROOT / 'reproducibility/workflows/regulatory_activity/run_tutorial.py'),
    "--data-root", str(DATA_ROOT),
    "--output-dir", str(CASE_OUTPUT),
    "--device", DEVICE,
    "--epochs", str(EPOCHS),
    "--seed", "42",
] + ['--dataset', 'elegans_tf', '--split', 'split_1', '--panel-size', '32', '--max-cells', '3000']
display_command = [
    "python", 'reproducibility/workflows/regulatory_activity/run_tutorial.py', "--data-root", "data/tutorials",
    "--output-dir", "outputs/tutorials/regulatory", "--device", DEVICE,
    "--epochs", str(EPOCHS), "--seed", "42",
] + ['--dataset', 'elegans_tf', '--split', 'split_1', '--panel-size', '32', '--max-cells', '3000']
print(" ".join(display_command))
completed = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if completed.returncode:
    print(completed.stdout)
    raise subprocess.CalledProcessError(completed.returncode, command)
manifest = json.loads((CASE_OUTPUT / "run_manifest.json").read_text())
print("Completed:", manifest["workflow"], "->", "outputs/tutorials/regulatory/run_manifest.json")


## Inspect newly generated outputs

In [ ]:
panel_path = CASE_OUTPUT / 'smith/panel_top32.csv'
ranking_path = sorted(CASE_OUTPUT.glob('smith/ranking/epoch_*.csv'))[-1]
metrics_path = CASE_OUTPUT / 'evaluation/metrics.tsv'
panel = pd.read_csv(panel_path, sep="\t" if panel_path.suffix == ".tsv" else ",")
ranking = pd.read_csv(ranking_path, sep="\t" if ranking_path.suffix == ".tsv" else ",")
metrics = pd.read_csv(metrics_path, sep="\t")
print("Generated panel:", panel_path.relative_to(CASE_OUTPUT))
display(panel.head(15))
print("Generated ranking:", ranking_path.relative_to(CASE_OUTPUT))
display(ranking.head(10))
display(metrics)


## Analyze this run

In [ ]:
ax = metrics.set_index("metric")["value"].plot(kind="bar", figsize=(9, 4.5), color="#2f6690")
ax.set_ylabel("value")
ax.set_title('Run the regulatory-activity workflow')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## Tutorial run versus manuscript run

The notebook uses one real TF split and tutorial-scale epochs. The manuscript result repeats the workflow across TF/miRNA splits, seeds, panel sizes and baselines.